In [1]:
!pip install google-api-python-client youtube-comment-downloader pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from googleapiclient.discovery import build
from youtube_comment_downloader import YoutubeCommentDownloader
import pandas as pd
import re

# -----------------------
# 1. API 설정
# -----------------------
API_KEY = "api_key"  # TODO: replace with your YouTube Data API key (do not commit real keys)

youtube = build("youtube", "v3", developerKey=API_KEY)

# -----------------------
# 2. 검색 키워드
# -----------------------
keywords = [
    "층간소음",
    "매트시공",
    "소음",
    "시끄러움",
    "시끄럽다",
    "발망치",
    "발소리",
    "쿵쿵",
    "의자 끄는 소리",
    "화장실",
    "강아지 짖는 소리",
    "문 쾅",
    "문콕",
    "새벽 드라이기",
    "청소기 소리",
    "차 소리",
    "경적",
    "덜컹",
    "아랫집 항의",
    "층간소음 스트레스",
    "층간소음 해결",
    "층간소음 신고",
    "층간소음 경찰",
    "층간소음 관리사무소",
    "층간소음 보복",
    "층간소음 참교육",
    "층간소음 발망치",
    "윗집 발망치",
    "윗집 소음",
    "새벽 층간소음",
    "아이 뛰는 소리 층간소음",
    "아파트 층간소음",
    "층간소음 방음매트",
    "층간소음 측정",
    "층간소음 민원",
    "층간소음 법",
    "층간소음 사건",
    "층간소음 이웃사이센터"
]

videos = []

# -----------------------
# 3. 유튜브 영상 검색
# -----------------------

seen_ids = set()
for keyword in keywords:

    request = youtube.search().list(
        q=keyword,
        part="id,snippet",
        maxResults=25,
        type="video"
    )

    response = request.execute()

    for item in response.get("items",[]):
        video_id = item.get("id", {}).get("videoId")
        if video_id and video_id not in seen_ids:
            title = item["snippet"]["title"]
            videos.append((video_id, title))
            seen_ids.add(video_id)

print("수집된 영상 수:", len(videos))


# -----------------------
# 4. 댓글 수집
# -----------------------
downloader = YoutubeCommentDownloader()

data = []

for video_id, title in videos:

    url = f"https://www.youtube.com/watch?v={video_id}"

    try:
        comments = downloader.get_comments_from_url(url)

        count = 0

        # -----------------------
        # 4. 댓글 수집 부분 수정
        # -----------------------
        for comment in comments:
            contents = comment["text"]
            date = comment["time"] # 예: "1년 전" 또는 "1 year ago"
            author = comment["author"]

            
            # 1. 한국어 '년' 또는 영어 'year'가 포함되어 있으면 제외 (1년 이상 된 데이터)
            if ("년" in date) or ("year" in date):
                match = re.search(r'\d+', date)
                if match:
                    years = int(match.group())

                    # 4년 이상 된 댓글은 건너뛰기 (3년 전까지만 수집)
                    if years > 3:
                        continue
        
            
        
            data.append({
                "title": title,
                "date": date,
                "author" : author,
                "contents": contents
                
            })
        
            count += 1
            if count == 200:
                break

    except:
        continue


# -----------------------
# 5. DataFrame 생성
# -----------------------
df = pd.DataFrame(data)

print(df.head())
print("총 댓글 수:", len(df))


# -----------------------
# 6. CSV 저장
# -----------------------
df.to_csv("층간소음_youtube.csv", index=False)

/Users/suuu/.pyenv/versions/da_env/lib/python3.9/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.18). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/suuu/.pyenv/versions/da_env/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/suuu/.pyenv/versions/da_env/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will 

수집된 영상 수: 626
                                               title   date      author  \
0  엘베 안 흉기에 40차례…층간 소음 가해? &quot;억울해&quot; / JTBC...  19분 전  @쿠루쿠루뚜르뜨루뜨   
1  엘베 안 흉기에 40차례…층간 소음 가해? &quot;억울해&quot; / JTBC...  29분 전    @김선우-v3v   
2  엘베 안 흉기에 40차례…층간 소음 가해? &quot;억울해&quot; / JTBC...  57분 전    @김나스-k9r   
3  엘베 안 흉기에 40차례…층간 소음 가해? &quot;억울해&quot; / JTBC...  1시간 전    @심종수-y7p   
4  엘베 안 흉기에 40차례…층간 소음 가해? &quot;억울해&quot; / JTBC...  1시간 전    @은채이-b9u   

                           contents  
0                         건설사가 어디임?  
1                      미친놈 부모가 젤 잘못  
2  가족들 그엘베 타는거 진짜 고역이겠다.. 어떻게 사냐 진짜  
3                   저런인가은정신병원영원히가둬라  
4                         대체 어느아파트?  
총 댓글 수: 51515


In [5]:
df.to_excel('층간소음_youtube.xlsx', index=False)

In [4]:
!pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
